In [71]:
import geopandas as gpd
import xml.etree.ElementTree as ET
from shapely.geometry import Point
from glob import glob

In [72]:
pfad = "./afis_xml/*.xml"

In [73]:
hf = []
lf = []
po = []

ET.register_namespace("", "http://www.adv-online.de/namespaces/adv/gid/7.1")
ET.register_namespace("gmd", "http://www.opengis.net/gml/3.2")

for f in glob(pfad):
    tree = ET.parse(f)
    root = tree.getroot()
    hf += root.findall(
        './/{http://www.adv-online.de/namespaces/adv/gid/7.1}AX_Hoehenfestpunkt')
    lf += root.findall(
        './/{http://www.adv-online.de/namespaces/adv/gid/7.1}AX_Lagefestpunkt')
    po += root.findall(
        './/{http://www.adv-online.de/namespaces/adv/gid/7.1}AX_PunktortAU')

In [74]:
len(hf), len(lf), len(po)

(490, 1419, 11733)

In [75]:
liste = {}
for i in po:
    idtag = i.find(
        './/{http://www.adv-online.de/namespaces/adv/gid/7.1}istTeilVon')
    if idtag is None:
        continue
    id = idtag.attrib['{http://www.w3.org/1999/xlink}href']
    if not id in liste:
        liste[id] = [i]
    else:
        liste[id].append(i)

In [76]:
for j in list(liste.items()):
    print(j[0])
    for i in j[1]:
        pt = i.find(
            './/{http://www.opengis.net/gml/3.2}Point')
        if ('srsName' in pt.attrib):
            print(pt.attrib['srsName'])
        else:
            # print(ET.tostring(pt, encoding='unicode'))
            print("Kein srsName")
        print(i.find(
            './/{http://www.opengis.net/gml/3.2}pos').text)

urn:adv:oid:DESHPDHK00001IzY
urn:adv:crs:DE_DHHN92_NH
21.450
Kein srsName
510290.200 6014081.090
urn:adv:crs:ETRS89_h
61.295
urn:adv:crs:DE_DHHN2016_NH
21.451
urn:adv:oid:DESHPDHK00001IA5
urn:adv:crs:DE_DHHN12_NOH
23.070
urn:adv:crs:DE_DHHN92_NH
23.040
urn:adv:crs:DE_DHDN_3GK3_SH210
510350.630 6016042.210
Kein srsName
510277.550 6014076.330
urn:adv:crs:DE_Bessel_h
25.220
urn:adv:crs:ETRS89_h
62.885
urn:adv:crs:DE_DHHN2016_NH
23.041
urn:adv:oid:DESHPDHK00001IAA
urn:adv:crs:DE_DHHN12_NOH
3.410
urn:adv:crs:DE_DHHN92_NH
3.300
urn:adv:crs:DE_DHDN_3GK3_SH210
501826.040 6007941.340
Kein srsName
501756.220 6005978.860
urn:adv:crs:DE_Bessel_h
5.080
urn:adv:crs:ETRS89_h
43.071
urn:adv:crs:DE_Soldner-Ostenfeld
-13449.280 -29666.460
urn:adv:crs:DE_DHHN2016_NH
3.301
urn:adv:oid:DESHPDHK00001IAK
urn:adv:crs:DE_DHHN12_NOH
0.940
urn:adv:crs:DE_DHHN92_NH
0.810
urn:adv:crs:DE_DHDN_3GK3_SH210
501835.980 6008079.630
Kein srsName
501766.160 6006117.100
urn:adv:crs:DE_Bessel_h
2.590
urn:adv:crs:ETRS89_h
40.

In [77]:
print(ET.tostring(lf[100], encoding='unicode'))

<AX_Lagefestpunkt xmlns="http://www.adv-online.de/namespaces/adv/gid/7.1" xmlns:gmd="http://www.opengis.net/gml/3.2" xmlns:ns2="http://www.w3.org/1999/xlink" gmd:id="DESHPDHK00001IMO">
					<gmd:identifier codeSpace="http://www.adv-online.de/">urn:adv:oid:DESHPDHK00001IMO</gmd:identifier>
					<lebenszeitintervall>
						<AA_Lebenszeitintervall>
							<beginnt>2011-03-30T14:41:22Z</beginnt>
						</AA_Lebenszeitintervall>
					</lebenszeitintervall>
					<modellart>
						<AA_Modellart>
							<advStandardModell>DFGM</advStandardModell>
						</AA_Modellart>
					</modellart>
					<punktkennung>172006310</punktkennung>
					<gemeinde>
						<AX_Gemeindekennzeichen>
							<land>01</land>
							<regierungsbezirk>0</regierungsbezirk>
							<kreis>51</kreis>
							<gemeinde>044</gemeinde>
						</AX_Gemeindekennzeichen>
					</gemeinde>
					<katasteramt>
						<AX_Dienststelle_Schluessel>
							<land>01</land>
							<stelle>0001</stelle>
						</AX_Dienststelle_Schluessel>
					</k

In [78]:
lagefestpunkte = []
for l in lf:
    id = l.find('{http://www.opengis.net/gml/3.2}identifier').text
    print(id)
    xy = None
    h = None
    for i in liste[id]:
        pt = i.find(
            './/{http://www.opengis.net/gml/3.2}Point')
        k = pt.find(
            './/{http://www.opengis.net/gml/3.2}pos').text
        if ('srsName' in pt.attrib):
            print(pt.attrib['srsName'], '\t-\t', k)
            if pt.attrib['srsName'] == 'urn:adv:crs:DE_DHHN2016_NH':
                h = k
            if pt.attrib['srsName'] == 'urn:adv:crs:DE_DHDN_3GK3_SH210':
                xy = k.split(' ')
        else:
            # print(ET.tostring(pt, encoding='unicode'))
            print("Kein srsName", '\t-\t', k)
    vermarkung = l.find(
        '{http://www.adv-online.de/namespaces/adv/gid/7.1}punktvermarkung').text
    relHoehe = l.find(
        '{http://www.adv-online.de/namespaces/adv/gid/7.1}relativeHoehe')
    if relHoehe is not None:
        relHoehe = relHoehe.text
    if (xy is not None) and (h is not None):
        lagefestpunkte.append([id, vermarkung, relHoehe, 3000000+float(xy[0]), float(
            xy[1]), float(h), Point(3000000+float(xy[0]), float(xy[1]), float(h))])

urn:adv:oid:DESHPDHK00001IA5
urn:adv:crs:DE_DHHN12_NOH 	-	 23.070
urn:adv:crs:DE_DHHN92_NH 	-	 23.040
urn:adv:crs:DE_DHDN_3GK3_SH210 	-	 510350.630 6016042.210
Kein srsName 	-	 510277.550 6014076.330
urn:adv:crs:DE_Bessel_h 	-	 25.220
urn:adv:crs:ETRS89_h 	-	 62.885
urn:adv:crs:DE_DHHN2016_NH 	-	 23.041
urn:adv:oid:DESHPDHK00001IAA
urn:adv:crs:DE_DHHN12_NOH 	-	 3.410
urn:adv:crs:DE_DHHN92_NH 	-	 3.300
urn:adv:crs:DE_DHDN_3GK3_SH210 	-	 501826.040 6007941.340
Kein srsName 	-	 501756.220 6005978.860
urn:adv:crs:DE_Bessel_h 	-	 5.080
urn:adv:crs:ETRS89_h 	-	 43.071
urn:adv:crs:DE_Soldner-Ostenfeld 	-	 -13449.280 -29666.460
urn:adv:crs:DE_DHHN2016_NH 	-	 3.301
urn:adv:oid:DESHPDHK00001IAK
urn:adv:crs:DE_DHHN12_NOH 	-	 0.940
urn:adv:crs:DE_DHHN92_NH 	-	 0.810
urn:adv:crs:DE_DHDN_3GK3_SH210 	-	 501835.980 6008079.630
Kein srsName 	-	 501766.160 6006117.100
urn:adv:crs:DE_Bessel_h 	-	 2.590
urn:adv:crs:ETRS89_h 	-	 40.582
urn:adv:crs:DE_DHHN2016_NH 	-	 0.811
urn:adv:oid:DESHPDHK00001IAR
urn:a

In [79]:
df_lf = gpd.GeoDataFrame(lagefestpunkte, columns=[
                         'id', 'vermarkung', 'relHoehe', 'x', 'y', 'z',  'geometry'])
df_lf.crs = 'EPSG:31463'
df_lf.to_file('sh.gpkg', driver='GPKG')